# GRIT projection-only — Kaggle T4 × 2

Notebook training nhận **bundle đã hoàn tất** từ `GRIT_Kaggle_T4x2_Prepare_Data_Projectors.ipynb`.
Chọn accelerator **GPU T4 × 2**, bật Internet, attach Dataset/Notebook Output chứa `bundle_manifest.json`.
Chạy lần lượt từ trên xuống; notebook gọi `scripts/run_grit_vllm.sh` từ checkout GRIT.

**Tình trạng bản giao:** backend đã được sửa cho projection-only và kiểm tra bằng regression tests CPU;
**chưa xác nhận GPU end-to-end**. Trước khi chạy, commit + push các thay đổi repo rồi điền SHA mới
vào `REPO_REF`. Không dùng commit cũ `d9bbce1b32881dc3161c7ab84d774e901aa4815e`.
Audit kiểm tra source và smoke kiểm tra runtime. Notebook gọi implementation trong repo.


Update yêu cầu: `d = AdamW_direction(g_task)`; MLP dùng `W += lr * (d @ P)`;
parameter khác dùng `theta += lr * d`. Moments nhận gradient chưa chiếu, LR nhân đúng một lần.
Không predictor, preservation backward, curvature/HVP, periodic weight repair, task KL hoặc LoRA.
Projector cố định FP32; `preserve_contexts.parquet` chỉ dùng để kiểm chứng provenance.
Không khẳng định bảo toàn mọi năng lực ngoài các hướng activation và module được chọn.

GPU 0: actor + policy vLLM; GPU 1: Qwen3Guard. Hai GPU là hai vùng VRAM riêng.
Output ở `/kaggle/working/grit_projection_only`; cần **Save Version và lưu output** trước khi kết thúc phiên.
Notebook không chứa secret. Sau full training/reload thành công, model tự upload lên `fushinguyenex/GRIT` bằng Kaggle Secret `HF_TOKEN`.

In [ ]:
# 1. Config — chỉ dùng stdlib; không import Torch trước cell pip/restart.
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, time

REPO_URL = "https://github.com/namdeptraivcd/GRIT.git"
REPO_REF = ""  # Điền SHA 40 ký tự của commit backend mới SAU KHI commit và push lên GitHub.
REPO = Path("/kaggle/working/GRIT")
OUT = Path("/kaggle/working/grit_projection_only")
INPUT_ROOT = Path("/kaggle/input")
BUNDLE_OVERRIDE = None  # Khi có nhiều bundle: điền đường dẫn folder cụ thể trong Input.
GUARD_MODEL = "Qwen/Qwen3Guard-Gen-0.6B"
GUARD_REVISION = None  # None: resolve SHA một lần rồi khóa vào model_lock.json.
RUN_FULL_TRAINING = False  # Bật sau khi smoke + resume đã pass.
CREATE_ARCHIVE = False    # Tùy chọn; cần thêm disk cho archive.
RUN_RELOAD = True
UPLOAD_TO_HUB = True      # Upload tự động sau full training + reload thành công.
HF_REPO_ID = "fushinguyenex/GRIT"
HF_TOKEN_SECRET = "HF_TOKEN"  # Kaggle Add-ons → Secrets; token có quyền Write vào repo.
RESUME_INPUT = None       # Folder run output cũ trong /kaggle/input; đọc cell resume trước.
CFG = dict(seed=66, train_batch_prompts=2, grpo_generations=4,
           micro_batch_responses=1, max_prompt_tokens=512, max_response_tokens=128,
           ppo_epochs=1, learning_rate=1e-6, weight_decay=0.0,
           clip_range=0.2, max_grad_norm=1.0, smoke_steps=2,
           full_training_steps=500, save_every=25, eval_every=25,
           actor_gpu=0, guard_gpu=1, rollout_gpu_memory=0.20,
           guard_gpu_memory=0.35, guard_port=52001,
           temperature=1.0, top_p=1.0, top_k=-1,
           lambda_pres=0.0, use_curvature=False, preservation_enable=False)
assert INPUT_ROOT.is_dir(), "Notebook này chạy trên Kaggle; attach bundle Input trước."
assert re.fullmatch(r"[0-9a-f]{40}", REPO_REF), "Pin repo bằng commit SHA 40 ký tự."
assert sys.version_info[:2] in [(3, 10), (3, 11), (3, 12)], "Stack dự kiến Python 3.10–3.12."
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "logs").mkdir(exist_ok=True)

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n")
    temp.replace(path)

def checked_files(root, files):
    root = Path(root).resolve()
    if not isinstance(files, dict) or not files:
        raise ValueError("Manifest phải có files: {relative_path: sha256}")
    for relative, expected in files.items():
        path = (root / relative).resolve()
        if not path.is_relative_to(root) or not path.is_file():
            raise ValueError(f"Artifact thiếu hoặc nằm ngoài bundle: {relative}")
        if not isinstance(expected, str) or not re.fullmatch(r"[0-9a-f]{64}", expected):
            raise ValueError(f"SHA256 không hợp lệ: {relative}")
        if sha256(path) != expected:
            raise ValueError(f"Checksum khác: {relative}; quay lại preprocessing.")

def load_json(path):
    return json.loads(Path(path).read_text())

print("Python:", sys.version.split()[0])
print("Working disk free GiB:", round(shutil.disk_usage(OUT).free / 2**30, 2))
print(subprocess.check_output(["nvidia-smi"], text=True))
assert shutil.disk_usage(OUT).free >= 15 * 2**30, "Cần tối thiểu 15 GiB trống; cache/model/checkpoint có thể cần thêm."


def inference_files(folder):
    folder = Path(folder)
    names = {"config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json",
             "special_tokens_map.json", "added_tokens.json", "vocab.json", "merges.txt",
             "tokenizer.model", "spiece.model", "chat_template.jinja", "model.safetensors.index.json"}
    result = sorted(p for p in folder.iterdir() if p.is_file() and (
        p.name in names or re.fullmatch(r"model(?:-\d+-of-\d+)?\.safetensors", p.name)))
    available = {p.name for p in result}
    assert {"config.json", "tokenizer_config.json"} <= available, "Thiếu model/tokenizer config."
    assert "tokenizer.json" in available or {"vocab.json", "merges.txt"} <= available, "Thiếu tokenizer assets."
    weights = {name for name in available if name.endswith(".safetensors")}
    assert weights, "Không tìm thấy merged safetensors weights."
    if "model.safetensors.index.json" in available:
        referenced = set(load_json(folder / "model.safetensors.index.json")["weight_map"].values())
        assert referenced == weights, "Shard index thiếu/thừa weights."
    else:
        assert weights == {"model.safetensors"}, "Sharded weights cần index."
    assert all(not p.is_symlink() and p.stat().st_size > 0 for p in result)
    return result


## 2. Resolve bundle và kiểm tra SHA256

Không chọn ngẫu nhiên khi nhiều bundle. Không tải/sampling/build lại dataset hoặc projector.
Schema bên dưới khớp notebook preprocessing đã cung cấp, gồm `files` là mapping tên file → SHA256.
Torch payload được load ở cell 6, sau khi stack đã cài xong và kernel đã restart.

In [ ]:
def resolve_bundle(input_root, override=None):
    root = Path(input_root).resolve()
    candidates = sorted(root.glob("**/bundle_manifest.json"))
    if override is not None:
        selected = Path(override).resolve() / "bundle_manifest.json"
        if not selected.is_relative_to(root) or selected not in candidates:
            raise ValueError("BUNDLE_OVERRIDE phải trỏ đến bundle trong Kaggle Input.")
    elif len(candidates) == 1:
        selected = candidates[0]
    else:
        raise ValueError(f"Cần đúng 1 bundle hoặc BUNDLE_OVERRIDE; tìm thấy {candidates}")
    manifest = load_json(selected)
    if manifest.get("status") != "complete":
        raise ValueError("Bundle chưa complete; quay lại notebook preprocessing.")
    required = {"task_train.parquet", "task_val.parquet", "preserve_prompts.parquet",
                "preserve_contexts.parquet", "qwen2_5_0_5b_context_projectors.pt",
                "revisions.json", "data_manifest.json", "manifest.json", "build_config.json"}
    if not required <= set(manifest.get("files", {})):
        raise ValueError(f"Thiếu file có checksum: {required - set(manifest.get('files', {}))}")
    checked_files(selected.parent, manifest["files"])
    return selected.parent, manifest

BUNDLE, BUNDLE_MANIFEST = resolve_bundle(INPUT_ROOT, BUNDLE_OVERRIDE)
REVISIONS = load_json(BUNDLE / "revisions.json")
CONTEXT_META = load_json(BUNDLE / "manifest.json")
DATA_META = load_json(BUNDLE / "data_manifest.json")
BUILD_CONFIG = load_json(BUNDLE / "build_config.json")
assert BUILD_CONFIG["config_hash"] == REVISIONS["config_hash"]
TASK_PROMPT_BUDGET = int(BUILD_CONFIG["task_max_prompt_tokens"])
CFG["max_prompt_tokens"] = TASK_PROMPT_BUDGET
MODEL_ID, MODEL_REVISION = REVISIONS["base_model"], REVISIONS["base_revision"]
assert MODEL_ID == "Qwen/Qwen2.5-0.5B-Instruct", "Notebook này kiểm chứng cho model 0.5B trong kịch bản."
assert re.fullmatch(r"[0-9a-f]{40}", MODEL_REVISION)
for meta in (CONTEXT_META, DATA_META, BUNDLE_MANIFEST):
    assert meta["base_model"] == MODEL_ID and meta["base_revision"] == MODEL_REVISION
    assert meta["tokenizer_sha256"] == CONTEXT_META["tokenizer_sha256"]
    assert meta["config_hash"] == REVISIONS["config_hash"]
checked_files(BUNDLE, DATA_META["files"])
checked_files(BUNDLE, {"preserve_contexts.parquet": CONTEXT_META["parquet_sha256"]})
assert CONTEXT_META["input_sha256"] == sha256(BUNDLE / "preserve_prompts.parquet")
assert BUNDLE_MANIFEST["preservation_contexts_sha256"] == CONTEXT_META["parquet_sha256"]
TASK_FILE, VAL_FILE = BUNDLE / "task_train.parquet", BUNDLE / "task_val.parquet"
PROJECTORS_PATH = BUNDLE / "qwen2_5_0_5b_context_projectors.pt"
atomic_json(OUT / "bundle_receipt.json", dict(bundle=str(BUNDLE),
            manifest_sha256=sha256(BUNDLE / "bundle_manifest.json"),
            base_model=MODEL_ID, base_revision=MODEL_REVISION, files=BUNDLE_MANIFEST["files"]))
print("Verified input:", BUNDLE)
print("Task prompt budget from preprocessing bundle:", TASK_PROMPT_BUDGET)


## 3. Clone và audit repo — trước khi tốn thời gian cài stack

Audit phát hiện các lỗi đã thấy trong checkout gốc: chiếu raw gradient rồi `optimizer.step`,
BF16 hard-code, thiếu GradScaler, reward bỏ prompt/parse substring, runner không chuyển tiếp CLI.
Audit source chỉ là gate phát hiện lỗi đã biết, **không thay thế integration test**.
Điền SHA commit backend đã sửa vào `REPO_REF` trước khi chạy. Commit cũ vẫn bị gate chặn.
Nhánh legacy preservation còn được giữ riêng; projection-only dùng ProjectedAdamW.

In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
assert (REPO / ".git").exists(), "Đường dẫn GRIT tồn tại nhưng không phải clone."
assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip() == REPO_URL
dirty = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, text=True)
assert not dirty.strip(), "Checkout có sửa đổi tracked files; dùng clone sạch để bảo đảm commit identity."
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", REPO_REF], cwd=REPO, check=True)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert COMMIT == REPO_REF
(OUT / "repo_commit.txt").write_text(COMMIT + "\n")

def audit_repo(root):
    root = Path(root)
    runner = (root / "scripts/run_grit_vllm.sh").read_text()
    actor = (root / "verl/verl/workers/actor/dp_actor.py").read_text()
    reward = (root / "scripts/nspo_vllm_reward.py").read_text()
    problems = []
    optimizer_path = root / "verl/verl/experimental/grit/optimizer.py"
    if not optimizer_path.is_file():
        problems.append("Repo thiếu optimizer chiếu AdamW task direction.")
    else:
        optimizer = optimizer_path.read_text()
        if "direction = direction @ projector" not in optimizer or 'p.add_(direction, alpha=group["lr"])' not in optimizer:
            problems.append("Chưa thấy projected direction và one-LR apply trong optimizer.")
    required = ["grit.enable=true", "grit.lambda_pres=0.0", "grit.use_curvature=false",
                "grit.preservation.enable=false", "grit.module_pattern=mlp",
                "grit.strict_projector_attach=true", "grit.require_projected_modules=true",
                "algorithm.use_kl_in_reward=false", "actor_rollout_ref.actor.use_kl_loss=false"]
    for setting in required:
        if setting not in runner:
            problems.append("Runner thiếu invariant: " + setting)
    if '"$@"' not in runner:
        problems.append("Runner chưa forward CLI overrides (\"$@\") vào main_ppo: FP16, seed, weight_decay, resume không áp dụng được.")
    if "ProjectedAdamW" not in actor or "and not self.projected_adamw" not in actor:
        problems.append("dp_actor chưa nối ProjectedAdamW hoặc chưa tách nhánh legacy raw-gradient projection.")
    if "dtype=torch.bfloat16" in actor:
        problems.append("dp_actor hard-code BF16 autocast; T4 cần FP16 + FP32 master weights.")
    if "GradScaler" not in actor and "grad_scaler" not in actor:
        problems.append("Chưa thấy actor GradScaler/unscale + xử lý nonfinite cho FP16.")
    if "zero_advantage" not in actor:
        problems.append("Chưa thấy skip optimizer state/update cho zero-advantage batch.")
    if 'if "unsafe" in label else 0.0' in reward:
        problems.append("Reward hiện tự gán Safe khi parse lỗi; cần structured Safety label và retry hữu hạn.")
    if "model_answer=response" in reward and 'del data_source, ground_truth, extra_info' in reward:
        problems.append("Reward bỏ user prompt, dùng template classifier cũ; cần template chính thức cho prompt + response.")
    # Những bằng chứng source này là cần thiết, chưa đủ để chứng minh đúng runtime.
    if "apply_chat_template" not in reward or "Controversial" not in reward:
        problems.append("Chưa thấy template Qwen3Guard và nhãn Controversial trong reward entrypoint.")
    return problems

REPO_PROBLEMS = audit_repo(REPO)
atomic_json(OUT / "repo_preflight.json", dict(commit=COMMIT,
            status="blocked" if REPO_PROBLEMS else "source_checks_passed",
            problems=REPO_PROBLEMS, gpu_end_to_end_verified=False))
for problem in REPO_PROBLEMS:
    print("BLOCKER:", problem)
assert not REPO_PROBLEMS, (
    "Repo chưa thực hiện đúng kịch bản. Sửa backend trong repo và pin REPO_REF mới; "
    "chi tiết: " + str(OUT / "repo_preflight.json"))


## 4. Cài stack pin và restart kernel

Cài vLLM trước, sau đó requirements Kaggle và dependency cơ bản của verl; không cài extras `verl[vllm]`
vì range của checkout cũ không nhận 0.9.2. Actor phải dùng SDPA/eager, không yêu cầu FlashAttention-2.
Pins chính: vLLM 0.9.2, Torch 2.7.0, torchvision 0.22.0, torchaudio 2.7.0, xFormers 0.0.30,
Transformers 4.53.3, Hub 0.34.4. Đây là stack ứng viên chưa được xác nhận trên Kaggle image thực.

Cell ghi marker và yêu cầu **Restart Session/Kernel**, sau đó chạy lại từ cell 1.
Lần chạy lại sẽ dùng marker để không pip lặp vô hạn. Nếu install/import lỗi, giữ log và sửa đúng
pin/xung đột đã báo; không tự nâng tất cả lên latest. `pip check` có thể báo package image Kaggle
không liên quan; notebook dừng để kiểm tra thay vì bỏ qua.

In [ ]:
assert not audit_repo(REPO), "Chưa qua repo audit."
STACK = {"vllm": "0.9.2", "torch": "2.7.0", "torchvision": "0.22.0",
         "torchaudio": "2.7.0", "xformers": "0.0.30", "transformers": "4.53.3",
         "huggingface-hub": "0.34.4"}
setup_path = OUT / "setup_state.json"
setup_key = digest(dict(stack=STACK, commit=COMMIT, python=sys.version.split()[0], executable=sys.executable))
setup_state = load_json(setup_path) if setup_path.exists() else {}
if setup_state.get("key") != setup_key:
    constraints = OUT / "stack-constraints.txt"
    constraints.write_text("\n".join(f"{k}=={v}" for k, v in STACK.items()) + "\nnumpy<2\n")
    pip = [sys.executable, "-m", "pip", "install", "-c", str(constraints)]
    commands = [
        pip + [f"{k}=={v}" for k, v in STACK.items()],
        pip + ["-r", str(REPO / "requirements-kaggle.txt")],
        # Base requirements only: no vLLM downgrade, no flash-attn build on T4.
        pip + ["codetiming", "dill", "hydra-core", "numpy<2", "peft==0.16.0",
               "pybind11", "pylatexenc", "ray[default]==2.47.1", "torchdata==0.11.0",
               "tensordict==0.8.3", "wandb", "tensorboard", "psutil", "pytest"],
        pip + ["-e", str(REPO / "verl"), "--no-deps"],
    ]
    with (OUT / "logs/setup.log").open("w") as log:
        for command in commands:
            print("Installing:", command[-3:], flush=True)
            result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
            if result.returncode:
                print((OUT / "logs/setup.log").read_text(errors="replace")[-10000:])
                raise RuntimeError("Pip thất bại; xem setup.log. Không tiếp tục bằng stack cũ.")
    atomic_json(setup_path, dict(key=setup_key, installer_pid=os.getpid()))
    raise RuntimeError("Cài đặt xong. RESTART KERNEL rồi chạy lại từ cell 1; không chạy cell import ngay.")
assert setup_state["installer_pid"] != os.getpid(), "Phải RESTART KERNEL sau pip."
import importlib.metadata as im
required_versions = {name: im.version(name) for name in STACK}
for name, expected in STACK.items():
    assert required_versions[name].split("+")[0] == expected, (name, required_versions[name], expected)
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
if pip_check.returncode:
    print("Warning: Kaggle base-image dependency conflicts are outside the GRIT stack:")
    print(pip_check.stdout.strip())
print("GRIT stack versions:", required_versions)
print("Setup marker matched; fresh kernel confirmed.")


In [ ]:
# 5. Package / hardware / T4 GPU smoke trong subprocess, không giữ CUDA context ở kernel.
import importlib.metadata as im
versions = {name: im.version(name) for name in STACK}
for name, expected in STACK.items():
    assert versions[name].split("+")[0] == expected, (name, versions[name], expected)
os.environ.update(VLLM_USE_V1="0", VLLM_ATTENTION_BACKEND="XFORMERS",
                  TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_XET="1",
                  HF_HUB_DOWNLOAD_TIMEOUT="120", HF_HUB_ETAG_TIMEOUT="120")
os.environ["PYTHONPATH"] = str(REPO) + os.pathsep + str(REPO / "verl")
sys.path[:0] = [str(REPO), str(REPO / "verl")]
import psutil
ram = psutil.virtual_memory()
print("RAM total/available GiB:", round(ram.total/2**30, 1), round(ram.available/2**30, 1))
assert ram.available >= 8 * 2**30, "Thiếu RAM khả dụng; mục tiêu host khoảng 32 GiB."
GPU_SMOKE = r"""
import json, torch, vllm, verl, transformers, xformers
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Bật Kaggle T4 x2"
gpus = []
for i in range(2):
    p = torch.cuda.get_device_properties(i)
    assert "T4" in p.name, f"Expected T4, got {p.name}"
    assert p.total_memory >= 14 * 2**30
    with torch.cuda.device(i):
        a = torch.randn(256, 256, device=f"cuda:{i}", dtype=torch.float16)
        assert torch.isfinite(a @ a.T).all()
        torch.cuda.synchronize()
    gpus.append(dict(index=i, name=p.name, capability=[p.major, p.minor], bytes=p.total_memory))
print(json.dumps(dict(gpus=gpus, cuda=torch.version.cuda)))
"""
gpu_text = subprocess.check_output([sys.executable, "-c", GPU_SMOKE], text=True)
print(gpu_text)
atomic_json(OUT / "environment.json", dict(versions=versions, python=sys.version,
            gpu_smoke_output=gpu_text, ram_total=ram.total, ram_available=ram.available))
(OUT / "requirements-freeze.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))


## 6. Model identity, schema và projector CPU

Base tải đúng SHA trong bundle. Guard resolve SHA một lần và reuse lock khi chạy lại.
Tokenizer fingerprint dùng đúng phép serialize của preprocessing, kể cả pad token.
Validator kiểm tra tất cả module MLP, dtype/shape/finite, metadata và context identity;
không fallback `missing=identity`. Không load model weights để validate shape.

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoConfig, AutoTokenizer
import torch
import pyarrow.parquet as pq

lock_path = OUT / "model_lock.json"
if lock_path.exists():
    MODEL_LOCK = load_json(lock_path)
    assert MODEL_LOCK["base_model"] == MODEL_ID and MODEL_LOCK["base_revision"] == MODEL_REVISION
    assert MODEL_LOCK["guard_model"] == GUARD_MODEL
    if GUARD_REVISION is not None:
        assert MODEL_LOCK["guard_revision"] == GUARD_REVISION
else:
    guard_sha = HfApi().model_info(GUARD_MODEL, revision=GUARD_REVISION or "main").sha
    MODEL_LOCK = dict(base_model=MODEL_ID, base_revision=MODEL_REVISION,
                      guard_model=GUARD_MODEL, guard_revision=guard_sha)
    atomic_json(lock_path, MODEL_LOCK)
assert re.fullmatch(r"[0-9a-f]{40}", MODEL_LOCK["guard_revision"])
patterns = ["*.json", "*.safetensors", "*.txt", "*.model", "*.tiktoken"]
MODEL_PATH = snapshot_download(MODEL_ID, revision=MODEL_REVISION, allow_patterns=patterns)
GUARD_PATH = snapshot_download(GUARD_MODEL, revision=MODEL_LOCK["guard_revision"], allow_patterns=patterns)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
fingerprint = digest(dict(vocab=tokenizer.get_vocab(), special_tokens=tokenizer.special_tokens_map,
                          chat_template=tokenizer.chat_template))
assert fingerprint == CONTEXT_META["tokenizer_sha256"], "Tokenizer không khớp preprocessing."
model_config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=False)
hidden_size = int(model_config.hidden_size)
intermediate_size = int(model_config.intermediate_size)
num_hidden_layers = int(model_config.num_hidden_layers)
expected_modules = {
    f"model.layers.{layer}.mlp.{part}": (intermediate_size if part == "down_proj" else hidden_size)
    for layer in range(num_hidden_layers)
    for part in ("gate_proj", "up_proj", "down_proj")
}

def validate_projectors(payload, expected, identity, context_hash):
    if not {"projectors", "metadata", "provenance"} <= set(payload):
        raise ValueError("Artifact phải có projectors/metadata/provenance.")
    ps, metadata, provenance = payload["projectors"], payload["metadata"], payload["provenance"]
    if not expected or set(ps) != set(expected) or set(metadata) != set(expected):
        raise ValueError("Thiếu/thừa module projector hoặc metadata so với model.")
    for key, value in identity.items():
        if provenance.get(key) != value:
            raise ValueError(f"Projector identity mismatch: {key}")
    if provenance.get("preservation_contexts_sha256") != context_hash:
        raise ValueError("Projector build từ context khác.")
    for name, dim in expected.items():
        p, meta = ps[name], metadata[name]
        if p.dtype != torch.float32 or tuple(p.shape) != (dim, dim) or not torch.isfinite(p).all():
            raise ValueError(f"Invalid FP32 projector: {name}")
        if not (0 <= meta["rank"] <= dim and meta["rank"] + meta["nullity"] == dim):
            raise ValueError(f"Invalid spectrum metadata: {name}")
        # Random-vector probe; tránh P@P cubic trên dense projector lớn.
        generator = torch.Generator(device="cpu").manual_seed(66)
        x = torch.randn(dim, 2, generator=generator)
        px = p @ x
        if not torch.allclose(p @ px, px, atol=2e-3, rtol=2e-3):
            raise ValueError(f"Idempotence probe failed: {name}")
        if not torch.allclose(p @ x, p.T @ x, atol=2e-3, rtol=2e-3):
            raise ValueError(f"Symmetry probe failed: {name}")
    return len(ps)

payload = torch.load(PROJECTORS_PATH, map_location="cpu", weights_only=True)
projector_count = validate_projectors(payload, expected_modules,
    dict(base_model=MODEL_ID, base_revision=MODEL_REVISION, tokenizer_sha256=fingerprint),
    CONTEXT_META["parquet_sha256"])
assert projector_count == BUNDLE_MANIFEST["projector_modules"]
assert payload["args"]["relative_threshold"] == 5e-4
assert payload["provenance"]["config_hash"] == REVISIONS["config_hash"]
projector_bytes = sum(p.numel() * p.element_size() for p in payload["projectors"].values())
print("Validated projector modules:", projector_count, "FP32 GiB:", round(projector_bytes / 2**30, 3))
print("Zero-nullity modules:", BUNDLE_MANIFEST.get("zero_nullity_modules", []))
del payload

task_keys = []
for path in (TASK_FILE, VAL_FILE):
    rows = pq.read_table(path).to_pylist()
    assert len(rows) == DATA_META["counts"][path.name] > 0
    keys = set()
    for row in rows:
        assert isinstance(row.get("raw_prompt"), str) and row["raw_prompt"].strip()
        ids = list(tokenizer.apply_chat_template([dict(role="user", content=row["raw_prompt"])],
                   tokenize=True, add_generation_prompt=True, return_dict=True)["input_ids"])
        assert ids == row["prompt_ids"], "Task tokenization drift."
        assert len(ids) <= CFG["max_prompt_tokens"], "Task vượt budget; không âm thầm truncate/sample lại."
        key = row["prompt_sha256"]
        assert key not in keys
        keys.add(key)
    task_keys.append(keys)
    print(path.name, len(rows), "prompts validated")
assert not (task_keys[0] & task_keys[1]), "Train/validation overlap."
contexts = pq.read_table(BUNDLE / "preserve_contexts.parquet").to_pylist()
assert len(contexts) == CONTEXT_META["rows"] == BUNDLE_MANIFEST["preservation_samples"]
for row in contexts:
    assert row["base_model"] == MODEL_ID and row["base_revision"] == MODEL_REVISION
    assert row["tokenizer_sha256"] == fingerprint
    ids, start = row["input_ids"], row["response_start"]
    assert 0 < start < len(ids) <= 2304
    assert row["response_mask"] == [0] * start + [1] * (len(ids) - start)
del contexts, rows
atomic_json(OUT / "artifact_validation.json", dict(status="passed", modules=projector_count,
    projector_bytes=projector_bytes, tokenizer_sha256=fingerprint))


## 7. Chuyển schema bằng adapter của repo và ghi run manifest

Giữ nguyên prompt source; adapter chỉ đổi raw prompt sang chat messages cho verl.
Không truyền preservation file vào runner. Training dùng response mask của verl: prompt/padding không
tham gia loss; EOS nằm trong response hợp lệ, response bị cắt vẫn dùng các token hợp lệ đã sinh.
Không thưởng refusal riêng. Safe=0, Unsafe/Controversial=-1; lỗi parse phải fail/retry có giới hạn.
Metrics refusal/parse errors, zero-advantage/nonfinite và độ lệch ratio cần có trong backend đã sửa.

In [ ]:
VERL_DATA = OUT / "verl_data"
VERL_DATA.mkdir(exist_ok=True)
adapter_sha = sha256(REPO / "scripts/prepare_verl_nspo_data.py")
for source, target_name in [(TASK_FILE, "train.parquet"), (VAL_FILE, "val.parquet")]:
    target, receipt = VERL_DATA / target_name, VERL_DATA / (target_name + ".json")
    signature = dict(input_sha256=sha256(source), adapter_sha256=adapter_sha)
    if target.exists() or receipt.exists():
        assert target.exists() and receipt.exists(), "Adapter cache chưa hoàn tất; xóa riêng cache lỗi hoặc dùng run mới."
        old = load_json(receipt)
        assert old["signature"] == signature and old["sha256"] == sha256(target)
    else:
        subprocess.run([sys.executable, str(REPO / "scripts/prepare_verl_nspo_data.py"),
                        "--input", str(source), "--output", str(target)], cwd=REPO, check=True)
        assert pq.read_metadata(target).num_rows == pq.read_metadata(source).num_rows
        atomic_json(receipt, dict(signature=signature, sha256=sha256(target)))

IDENTITY = dict(repo_commit=COMMIT, models=MODEL_LOCK, config=CFG,
    bundle_manifest_sha256=sha256(BUNDLE / "bundle_manifest.json"),
    projectors_sha256=sha256(PROJECTORS_PATH), tokenizer_sha256=fingerprint,
    versions=versions, algorithm="adamw_task_direction_then_right_project",
    reward=dict(Safe=0, Unsafe=-1, Controversial=-1), module_pattern="mlp")
RUN_ID = digest(IDENTITY)
manifest_path = OUT / "run_manifest.json"
if manifest_path.exists():
    assert load_json(manifest_path)["identity"] == IDENTITY, "Run identity thay đổi; chọn OUT mới."
else:
    atomic_json(manifest_path, dict(identity=IDENTITY, run_id=RUN_ID, status="preflight_passed",
                                   gpu_end_to_end_verified=False, commands=[]))
print("Run identity:", RUN_ID)


## 8. Runner, log, timeout và checkpoint receipts

Notebook chỉ điều phối process của runner và lưu file, không triển khai actor/RPC/weight sync.
Backend đã sửa phải chuyển tiếp CLI overrides ở cuối lệnh `main_ppo.py`.
Các override này cố định FP16 rollout, FP32 master, FP16 FSDP, weight decay=0, seed=66,
sequence mean loss và checkpoint state. Audit không tự chứng minh GradScaler hoạt động đúng.

Cell chạy độc quyền bằng file lock. Khi interrupt, gửi SIGINT và cho runner 20 giây thoát;
sau đó chỉ cleanup process group/descendants do cell tạo. Chỉ checkpoint đã finalize mới dùng để resume;
không hứa lưu được step đang dở. Log guard và sampled VRAM được giữ riêng. VRAM lấy mẫu mỗi 5 giây
có thể bỏ sót peak ngắn; dùng metrics allocator của worker nếu cần peak chính xác.

In [ ]:
import contextlib, fcntl, signal, socket, uuid

def complete_checkpoints(folder):
    found = []
    for p in Path(folder).glob("global_step_*"):
        if not p.is_dir() or not re.fullmatch(r"global_step_\d+", p.name):
            continue
        required = [p / "data.pt", p / "actor/model_world_size_1_rank_0.pt",
                    p / "actor/optim_world_size_1_rank_0.pt",
                    p / "actor/extra_state_world_size_1_rank_0.pt",
                    p / "actor/huggingface/config.json"]
        if all(f.is_file() and f.stat().st_size > 0 for f in required):
            found.append(p)
    return sorted(found, key=lambda p: int(p.name.rsplit("_", 1)[1]))

def checkpoint_receipt(checkpoint, create=False):
    checkpoint = Path(checkpoint)
    receipt = checkpoint / "notebook_receipt.json"
    if create:
        files = {str(p.relative_to(checkpoint)): sha256(p) for p in sorted(checkpoint.rglob("*"))
                 if p.is_file() and p != receipt and not p.name.endswith(".tmp")}
        atomic_json(receipt, dict(run_id=RUN_ID, identity=IDENTITY, files=files,
                                 step=int(checkpoint.name.rsplit("_", 1)[1])))
    value = load_json(receipt)
    assert value["run_id"] == RUN_ID and value["identity"] == IDENTITY, "Resume identity mismatch."
    checked_files(checkpoint, value["files"])
    return value

def training_command(stage, steps, save_every, eval_every, resume=None):
    folder = OUT / ("smoke" if stage.startswith("smoke") else "train") / "checkpoints"
    overrides = [
        "actor_rollout_ref.model.use_remove_padding=false",
        "actor_rollout_ref.model.enable_gradient_checkpointing=true",
        "++actor_rollout_ref.actor.fsdp_config.model_dtype=fp32",
        "++actor_rollout_ref.actor.fsdp_config.mixed_precision.param_dtype=fp16",
        "++actor_rollout_ref.actor.fsdp_config.mixed_precision.reduce_dtype=fp32",
        "++actor_rollout_ref.actor.fsdp_config.mixed_precision.buffer_dtype=fp32",
        "++actor_rollout_ref.actor.fsdp_config.use_orig_params=true",
        "++actor_rollout_ref.actor.fsdp_config.seed=66",
        "++actor_rollout_ref.model.override_config._attn_implementation=sdpa",
        "++actor_rollout_ref.model.override_config.use_cache=false",
        "actor_rollout_ref.rollout.dtype=float16",
        "actor_rollout_ref.actor.optim.lr=" + str(CFG["learning_rate"]),
        "actor_rollout_ref.actor.optim.weight_decay=0.0",
        "actor_rollout_ref.actor.ppo_epochs=1",
        "actor_rollout_ref.actor.clip_ratio=0.2",
        "actor_rollout_ref.actor.grad_clip=1.0",
        "actor_rollout_ref.actor.loss_agg_mode=seq-mean-token-mean",
        "actor_rollout_ref.actor.checkpoint.save_contents=[model,optimizer,extra]",
        "actor_rollout_ref.actor.checkpoint.load_contents=[model,optimizer,extra]",
        "++data.seed=66", "++actor_rollout_ref.rollout.seed=66",
        "trainer.max_actor_ckpt_to_keep=2",
        "trainer.resume_mode=" + ("resume_path" if resume else "disable"),
        "++ray_kwargs.ray_init.address=local",
        "++ray_kwargs.ray_init._temp_dir=/tmp/grit-ray-" + uuid.uuid4().hex[:12],
    ]
    if resume is not None:
        checkpoint_receipt(resume)
        overrides.append("trainer.resume_from_path=" + str(resume))
    env = os.environ.copy()
    env.pop("PRESERVE_FILE", None)
    env.update({k: str(v) for k, v in dict(
        PYTHON_BIN=sys.executable, MODEL_PATH=MODEL_PATH, SAFETY_MODEL_PATH=GUARD_PATH,
        TASK_FILE=TASK_FILE, VAL_FILE=VAL_FILE, PROJECTORS_PATH=PROJECTORS_PATH,
        OUTPUT_DIR=folder, VERL_DATA_DIR=VERL_DATA, MAX_STEPS=steps,
        SAVE_STEPS=save_every, EVAL_STEPS=eval_every,
        TRAIN_BATCH_SIZE=CFG["train_batch_prompts"], MICRO_BATCH_SIZE=CFG["micro_batch_responses"],
        GRPO_GENERATIONS=CFG["grpo_generations"], MAX_PROMPT_LENGTH=CFG["max_prompt_tokens"],
        MAX_RESPONSE_LENGTH=CFG["max_response_tokens"], ROLLOUT_TEMPERATURE=1, ROLLOUT_TOP_P=1,
        ACTOR_GPU=0, GUARD_GPU=1, GUARD_PORT=CFG["guard_port"],
        GUARD_BASE_URL=f"http://127.0.0.1:{CFG['guard_port']}/v1",
        GUARD_LOG=OUT / "logs" / (stage + "-qwen3guard_vllm.log"), START_GUARD_SERVER=1,
        GUARD_GPU_MEMORY_UTILIZATION=CFG["guard_gpu_memory"],
        ROLLOUT_GPU_MEMORY_UTILIZATION=CFG["rollout_gpu_memory"],
        NSPO_GUARD_TIMEOUT=120, NSPO_GUARD_WORKERS=4, PYTHONUNBUFFERED=1,
        VLLM_USE_V1=0, VLLM_ATTENTION_BACKEND="XFORMERS").items()})
    return ["bash", str(REPO / "scripts/run_grit_vllm.sh"), *overrides], env, folder

def run_stage(stage, steps, save_every, eval_every, resume=None, timeout_hours=10):
    assert not audit_repo(REPO), "Repo contract chưa pass."
    assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == COMMIT
    assert not subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, text=True).strip()
    command, env, folder = training_command(stage, steps, save_every, eval_every, resume)
    folder.mkdir(parents=True, exist_ok=True)
    if resume is None:
        assert not list(folder.glob("global_step_*")), "Có checkpoint cũ: resume rõ ràng hoặc chọn OUT mới."
    with (OUT / "runner.lock").open("a+") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        with socket.socket() as sock:
            assert sock.connect_ex(("127.0.0.1", CFG["guard_port"])) != 0, "Guard port đang được dùng."
        record = load_json(manifest_path)
        # Ghi chỉ env điều khiển run, không dump os.environ hoặc secret.
        safe_keys = ["MODEL_PATH", "SAFETY_MODEL_PATH", "TASK_FILE", "VAL_FILE", "PROJECTORS_PATH",
                     "OUTPUT_DIR", "MAX_STEPS", "SAVE_STEPS", "EVAL_STEPS", "ACTOR_GPU", "GUARD_GPU"]
        record["commands"].append(dict(stage=stage, argv=command, env={k: env[k] for k in safe_keys}))
        record["status"] = "running:" + stage
        atomic_json(manifest_path, record)
        logfile = OUT / "logs" / (stage + ".log")
        descendants, process = {}, None
        start, offset = time.monotonic(), 0
        gpu_samples = []
        try:
            with logfile.open("w") as log:
                process = subprocess.Popen(command, cwd=REPO, env=env, stdout=log,
                                           stderr=subprocess.STDOUT, start_new_session=True)
                parent = psutil.Process(process.pid)
                while process.poll() is None:
                    with contextlib.suppress(psutil.Error):
                        for child in parent.children(recursive=True):
                            descendants[(child.pid, child.create_time())] = child
                    if time.monotonic() - start > timeout_hours * 3600:
                        raise TimeoutError(f"{stage} timeout; xem {logfile}")
                    time.sleep(5)
                    with logfile.open() as reader:
                        reader.seek(offset)
                        chunk = reader.read()
                        offset = reader.tell()
                    if chunk:
                        print(chunk[-6000:], end="", flush=True)
                    sample = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,utilization.gpu",
                                             "--format=csv,noheader,nounits"], capture_output=True, text=True)
                    gpu_samples.append(dict(elapsed=time.monotonic()-start, values=sample.stdout.strip()))
                if process.returncode:
                    raise RuntimeError(f"Runner exit {process.returncode}; xem {logfile} và guard log.")
        finally:
            if process is not None:
                if process.poll() is None:
                    with contextlib.suppress(ProcessLookupError):
                        os.killpg(process.pid, signal.SIGINT)
                    with contextlib.suppress(subprocess.TimeoutExpired):
                        process.wait(timeout=20)
                with contextlib.suppress(ProcessLookupError):
                    os.killpg(process.pid, signal.SIGTERM)
                for child in descendants.values():
                    with contextlib.suppress(psutil.Error):
                        child.terminate()
                _, alive = psutil.wait_procs(list(descendants.values()), timeout=10)
                for child in alive:
                    with contextlib.suppress(psutil.Error):
                        child.kill()
                with contextlib.suppress(subprocess.TimeoutExpired):
                    process.wait(timeout=10)
                if process.poll() is None:
                    process.kill()
                    process.wait(timeout=5)
            atomic_json(OUT / "logs" / (stage + "-resources.json"),
                        dict(elapsed_seconds=time.monotonic()-start, gpu_samples=gpu_samples,
                             peak_note="nvidia-smi sampling, not exact allocator peak"))
            record = load_json(manifest_path)
            record["status"] = "runner_exited:" + stage if process and process.returncode == 0 else "interrupted_or_failed:" + stage
            atomic_json(manifest_path, record)
            # Marker được verl ghi sau model/optimizer/extra/data; giữ save đã finalize khi interrupt.
            marker = folder / "latest_checkpointed_iteration.txt"
            if marker.is_file() and marker.read_text().strip().isdigit():
                finalized_step = int(marker.read_text().strip())
                for checkpoint in complete_checkpoints(folder):
                    if int(checkpoint.name.rsplit("_", 1)[1]) <= finalized_step:
                        checkpoint_receipt(checkpoint, create=True)
        # Finalize receipts only after runner exits successfully and writes its completion marker.
        marker = folder / "latest_checkpointed_iteration.txt"
        assert marker.is_file(), "Thiếu completion marker verl."
        latest_step = int(marker.read_text().strip())
        checkpoints = complete_checkpoints(folder)
        assert checkpoints and int(checkpoints[-1].name.rsplit("_", 1)[1]) == latest_step == steps
        for checkpoint in checkpoints:
            checkpoint_receipt(checkpoint, create=True)
        return checkpoints[-1], logfile


## 9. Smoke 2 steps → restart workers → resume thêm 1 step

Đây là gate bắt buộc trước 500 steps. Guard readiness do runner kiểm tra qua localhost.
Source audit phải pass; smoke phải có projector count > 0 và checkpoint đủ model/optimizer/extra/data.
Metrics backend cần ghi: `grit/zero_advantage_batch`, `grit/nonfinite_skipped`, `guard/parse_errors`,
`guard/refusal_rate`, `grit/pre_update_ratio_max_abs_error`. Backend mới ghi các metric này; commit cũ không đáp ứng. Ratio tolerance ban đầu 0.05 là ngưỡng smoke cần đo lại;
không suy ra đúng sync chỉ từ checkpoint hoặc count. Nếu batch chỉ có reward đồng nhất, smoke có thể
chưa chứng minh learning; dừng full run và kiểm tra dữ liệu/reward trước.

In [ ]:
def metric_values(text, name):
    clean = re.sub(r"\x1b\[[0-9;]*m", "", text)
    number = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
    return [float(x) for x in re.findall(re.escape(name) + r"[\s'\"]*[:=][\s]*[\[ ]*(" + number + ")", clean)]

def validate_smoke_log(path):
    text = Path(path).read_text(errors="replace")
    assert "Guard server is ready" in text, "Chưa có bằng chứng guard ready."
    for name in ["grit/attached_projector_count", "grit/projected_module_count"]:
        values = metric_values(text, name)
        assert values and min(values) > 0, f"Thiếu hoặc zero {name}"
    for name in ["grit/zero_advantage_batch", "grit/nonfinite_skipped", "guard/parse_errors", "guard/refusal_rate"]:
        assert metric_values(text, name), f"Backend chưa xuất metric bắt buộc: {name}"
    assert min(metric_values(text, "grit/zero_advantage_batch")) == 0, "Chưa có batch có tín hiệu học."
    assert max(metric_values(text, "grit/nonfinite_skipped")) == 0, "FP16 smoke có nonfinite."
    assert max(metric_values(text, "guard/parse_errors")) == 0, "Guard có parse errors."
    ratio = metric_values(text, "grit/pre_update_ratio_max_abs_error")
    assert ratio and max(ratio) <= 0.05, "Chưa chứng minh vLLM/PyTorch ratio gần 1."
    return {name: metric_values(text, name) for name in [
        "grit/attached_projector_count", "grit/projected_module_count",
        "grit/zero_advantage_batch", "grit/nonfinite_skipped",
        "guard/parse_errors", "guard/refusal_rate", "grit/pre_update_ratio_max_abs_error"]}

smoke_receipt = OUT / "smoke_passed.json"
if smoke_receipt.exists():
    prior = load_json(smoke_receipt)
    assert prior["run_id"] == RUN_ID
    checkpoint_receipt(OUT / prior["checkpoint"])
    print("Reusing verified smoke + resume receipt.")
else:
    smoke_dir = OUT / "smoke/checkpoints"
    existing = complete_checkpoints(smoke_dir)
    if existing:
        # Rerun chỉ reuse checkpoint có receipt sau runner success; partial save không được dùng.
        latest = existing[-1]
        checkpoint_receipt(latest)
        assert latest.name in {"global_step_2", "global_step_3"}
        smoke_metrics = validate_smoke_log(OUT / "logs/smoke.log")
    else:
        latest, log = run_stage("smoke", CFG["smoke_steps"], 1, 1, timeout_hours=1)
        smoke_metrics = validate_smoke_log(log)
    if latest.name == "global_step_2":
        latest, resume_log = run_stage("smoke-resume", 3, 1, 1, resume=latest, timeout_hours=1)
    else:
        resume_log = OUT / "logs/smoke-resume.log"
    resume_text = resume_log.read_text(errors="replace")
    assert "Resuming from" in resume_text and "global_step_2" in resume_text
    resume_metrics = validate_smoke_log(resume_log)
    atomic_json(smoke_receipt, dict(run_id=RUN_ID, checkpoint=str(latest.relative_to(OUT)),
                                   smoke=smoke_metrics, resume=resume_metrics))
print("SMOKE + RESTART/RESUME PASSED; full training có thể bật ở config.")


## 10. Full training 500 steps / resume output đã lưu

Full run ở `train/checkpoints`, tách smoke để không mang scheduler horizon 2/3 steps sang run 500.
Nếu tiếp tục phiên cũ, attach output đã Save Version, đặt `RESUME_INPUT` là folder run cũ và đặt
`GUARD_REVISION` đúng `model_lock.json` cũ **trước khi chạy cell 6**. Config/package/commit phải trùng.
Chỉ copy checkpoint mới nhất đã có receipt và checksum vào Working; Input vẫn read-only.
Không auto dùng checkpoint khi identity khác. Runner giữ tối đa hai actor checkpoints; notebook chỉ dọn
folder cũ có receipt đúng run sau khi checkpoint mới đã finalize.

In [ ]:
TRAIN_DIR = OUT / "train/checkpoints"
if RUN_FULL_TRAINING:
    assert load_json(smoke_receipt)["run_id"] == RUN_ID
    checkpoint_receipt(OUT / load_json(smoke_receipt)["checkpoint"])
    TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    if RESUME_INPUT is not None:
        source_run = Path(RESUME_INPUT).resolve()
        assert source_run.is_relative_to(INPUT_ROOT.resolve())
        assert load_json(source_run / "run_manifest.json")["identity"] == IDENTITY
        candidates = complete_checkpoints(source_run / "train/checkpoints")
        assert candidates, "Không có checkpoint hoàn tất trong resume Input."
        source = candidates[-1]
        checkpoint_receipt(source)
        destination = TRAIN_DIR / source.name
        if not destination.exists():
            size = sum(p.stat().st_size for p in source.rglob("*") if p.is_file())
            assert shutil.disk_usage(OUT).free > size + 5 * 2**30
            partial = destination.with_name(destination.name + ".copying")
            assert not partial.exists(), "Có bản copy dở; kiểm tra trước khi chạy lại."
            shutil.copytree(source, partial)
            checked_files(partial, load_json(partial / "notebook_receipt.json")["files"])
            partial.rename(destination)
        checkpoint_receipt(destination)
    checkpoints = complete_checkpoints(TRAIN_DIR)
    resume = checkpoints[-1] if checkpoints else None
    if resume is not None:
        checkpoint_receipt(resume)
    if resume is None or int(resume.name.rsplit("_", 1)[1]) < CFG["full_training_steps"]:
        TRAIN_CHECKPOINT, TRAIN_LOG = run_stage("train", CFG["full_training_steps"],
            CFG["save_every"], CFG["eval_every"], resume=resume)
    else:
        TRAIN_CHECKPOINT = resume
        print("Full target đã đạt:", resume)
    # Chỉ dọn folder thuộc run này có receipt; không đụng checkpoint chưa có ownership receipt.
    owned = sorted([p for p in TRAIN_DIR.glob("global_step_*")
                    if re.fullmatch(r"global_step_\d+", p.name) and (p / "notebook_receipt.json").is_file()
                    and load_json(p / "notebook_receipt.json").get("run_id") == RUN_ID],
                   key=lambda p: int(p.name.rsplit("_", 1)[1]))
    for old in owned[:-2]:
        shutil.rmtree(old)
    record = load_json(manifest_path)
    record.update(status="full_training_finished", full_checkpoint=str(TRAIN_CHECKPOINT.relative_to(OUT)))
    atomic_json(manifest_path, record)
else:
    print("Full training chưa bật. Sau smoke, đặt RUN_FULL_TRAINING=True rồi chạy lại cell này.")


## 11. Reload checkpoint theo format verl → generate kiểm tra

Dùng model merger của repo; checkpoint FSDP không phải Hugging Face model trực tiếp.
Đây là kiểm tra reload/generation; validation safety trong training vẫn do runner thực hiện.
Không coi một câu trả lời là benchmark hoặc bằng chứng bảo toàn general capability.

In [ ]:
if RUN_RELOAD:
    selected = (complete_checkpoints(TRAIN_DIR) or complete_checkpoints(OUT / "smoke/checkpoints"))[-1]
    checkpoint_receipt(selected)
    export_dir = OUT / "exports" / (selected.parent.parent.name + "-" + selected.name)
    if not (export_dir / "reload_passed.json").exists() or "files" not in load_json(export_dir / "reload_passed.json"):
        subprocess.run([sys.executable, "-m", "verl.model_merger", "merge", "--backend", "fsdp",
                        "--local_dir", str(selected / "actor"), "--target_dir", str(export_dir)],
                       cwd=REPO, check=True)
        RELOAD = r"""
import json, sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
path = sys.argv[1]
tok = AutoTokenizer.from_pretrained(path, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16,
        attn_implementation="sdpa", trust_remote_code=False).to("cuda:0").eval()
prompt = "Explain why the sky appears blue in two sentences."
inputs = tok.apply_chat_template([dict(role="user", content=prompt)],
        add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda:0")
with torch.inference_mode():
    output = model.generate(**inputs, do_sample=False, max_new_tokens=64,
                            pad_token_id=tok.eos_token_id)
response = tok.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
assert response.strip(), "Empty reload generation"
print(json.dumps(dict(prompt=prompt, response=response), ensure_ascii=False))
"""
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = "0"
        output = subprocess.check_output([sys.executable, "-c", RELOAD, str(export_dir)], env=env, text=True)
        (export_dir / "reload_generation.txt").write_text(output)
        atomic_json(export_dir / "reload_passed.json", dict(run_id=RUN_ID,
                    checkpoint=str(selected.relative_to(OUT)), generation_file="reload_generation.txt",
                    files={p.name: sha256(p) for p in inference_files(export_dir)}))
    print("Reload result:", (export_dir / "reload_generation.txt").read_text())


## 12. Tự upload model đã train lên Hugging Face

Đích: [fushinguyenex/GRIT](https://huggingface.co/fushinguyenex/GRIT).
Trong Kaggle **Add-ons → Secrets**, thêm `HF_TOKEN` có quyền **Write** vào repo và bật quyền truy cập
cho notebook. Token chỉ dùng trong bộ nhớ khi upload, không in ra hoặc lưu trong artifact.

`UPLOAD_TO_HUB=True` mặc định theo yêu cầu. Cell chỉ publish khi **full training đạt số step cấu hình**,
manifest đúng run và checkpoint tương ứng đã reload/generate thành công. Smoke không được publish.
Model/tokenizer được upload vào `runs/<run_id>/global_step_<N>/` để phân biệt các lần train.
Có `training_metadata.json` và model card kèm lệnh `from_pretrained(..., subfolder=...)`.
Chỉ upload inference model/tokenizer cùng metadata đã chọn; checkpoint optimizer để resume tiếp tục
được giữ trong Kaggle output. Repo phải tồn tại và token phải truy cập được; notebook giữ nguyên visibility.

Upload lỗi: sửa token/quyền/mạng rồi chạy lại **riêng cell này**, không cần train lại.
Thành công sẽ có commit URL và receipt local `hub_upload.json`; chạy lại sẽ kiểm tra receipt đó.
Dùng [Hugging Face Hub upload API 0.34.4](https://huggingface.co/docs/huggingface_hub/v0.34.4/en/guides/upload).

In [ ]:
def publish_trained_model(api_factory, token_provider):
    if not UPLOAD_TO_HUB or not RUN_FULL_TRAINING:
        print("Chưa upload: cần UPLOAD_TO_HUB=True và full training thành công.")
        return None
    assert HF_REPO_ID == "fushinguyenex/GRIT", "Đích upload khác repo được yêu cầu."
    record = load_json(manifest_path)
    assert record["identity"] == IDENTITY and record["run_id"] == RUN_ID
    assert record["status"] == "full_training_finished", "Full training chưa hoàn tất."
    checkpoint = TRAIN_DIR / f"global_step_{CFG['full_training_steps']}"
    assert record["full_checkpoint"] == str(checkpoint.relative_to(OUT))
    checkpoint_receipt(checkpoint)
    model_dir = OUT / "exports" / ("train-" + checkpoint.name)
    reload_state = load_json(model_dir / "reload_passed.json")
    assert reload_state["run_id"] == RUN_ID
    assert reload_state["checkpoint"] == str(checkpoint.relative_to(OUT)), "Reload receipt khác checkpoint."
    files = {p.name: sha256(p) for p in inference_files(model_dir)}
    assert reload_state.get("files") == files, "Model đổi sau reload hoặc receipt cũ; chạy lại cell reload."
    subfolder = f"runs/{RUN_ID}/{checkpoint.name}"
    metadata = dict(run_id=RUN_ID, repo_commit=COMMIT, base_model=MODEL_ID,
        base_revision=MODEL_REVISION, training_steps=CFG["full_training_steps"], config=CFG,
        projectors_sha256=IDENTITY["projectors_sha256"], tokenizer_sha256=IDENTITY["tokenizer_sha256"],
        bundle_manifest_sha256=IDENTITY["bundle_manifest_sha256"],
        model_lock=MODEL_LOCK, versions=versions, model_files_sha256=files,
        algorithm="GRIT projection-only: AdamW task direction right-projected on fixed MLP projectors",
        preservation_correction=False, curvature=False, reload_generation_passed=True,
        evaluation_note="Reload generation passed; no claim of general-capability preservation.")
    atomic_json(model_dir / "training_metadata.json", metadata)
    card = f'''
---
base_model: {MODEL_ID}
library_name: transformers
pipeline_tag: text-generation
tags:
- grit
- projection-only
- grpo
---
# GRIT projection-only — {checkpoint.name}

Base revision: `{MODEL_REVISION}`. GRIT repo commit: `{COMMIT}`.
Training steps: {CFG['full_training_steps']}. Run ID: `{RUN_ID}`.
Uses fixed MLP projectors on AdamW task directions; no preservation correction or curvature.
Reload/generation passed. This is not evidence that all general capabilities are preserved.
See `training_metadata.json` for configuration, model revisions and artifact hashes.

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
repo = "{HF_REPO_ID}"
subfolder = "{subfolder}"
tokenizer = AutoTokenizer.from_pretrained(repo, subfolder=subfolder)
model = AutoModelForCausalLM.from_pretrained(repo, subfolder=subfolder)
```

This folder contains inference weights and tokenizer. Resume optimizer/trainer state is in Kaggle output.
'''
    (model_dir / "README.md").write_text(card.strip() + "\n")
    upload_names = sorted([*files, "training_metadata.json", "README.md"])
    upload_hashes = {name: sha256(model_dir / name) for name in upload_names}
    receipt_path = OUT / "hub_upload.json"
    token = token_provider()
    assert isinstance(token, str) and token.strip(), "Kaggle Secret HF_TOKEN rỗng."
    api = api_factory(token=token.strip())
    try:
        # Không create repo hoặc đổi public/private; kiểm tra repo người dùng chỉ định.
        api.repo_info(repo_id=HF_REPO_ID, repo_type="model")
        prior = load_json(receipt_path) if receipt_path.exists() else None
        if prior is not None:
            assert prior["repo_id"] == HF_REPO_ID and prior["run_id"] == RUN_ID
            assert prior["subfolder"] == subfolder and prior["files"] == upload_hashes
            api.repo_info(repo_id=HF_REPO_ID, repo_type="model", revision=prior["commit_sha"])
            print("Model đã upload:", prior["model_url"])
            return prior
        commit = api.upload_folder(repo_id=HF_REPO_ID, repo_type="model", revision="main",
            folder_path=str(model_dir), path_in_repo=subfolder, allow_patterns=upload_names,
            commit_message=f"Add GRIT projection-only {checkpoint.name} ({RUN_ID[:12]})")
        # Verify file presence at the immutable commit, not a moving main revision.
        remote = set(api.list_repo_files(repo_id=HF_REPO_ID, repo_type="model", revision=commit.oid))
        assert {f"{subfolder}/{name}" for name in upload_names} <= remote, "Upload thiếu file; chưa ghi success receipt."
        result = dict(repo_id=HF_REPO_ID, run_id=RUN_ID, subfolder=subfolder,
            commit_sha=commit.oid, commit_url=commit.commit_url, files=upload_hashes,
            model_url=f"https://huggingface.co/{HF_REPO_ID}/tree/{commit.oid}/{subfolder}")
        atomic_json(receipt_path, result)
        print("Upload thành công:", result["model_url"])
        print("Commit:", result["commit_url"])
        print(f"Load với repo_id={HF_REPO_ID!r}, subfolder={subfolder!r}, revision={commit.oid!r}")
        return result
    finally:
        # Không login() / ghi token vào disk hoặc metadata.
        del token, api

def kaggle_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(HF_TOKEN_SECRET)
    except Exception:
        raise RuntimeError("Thêm và bật Kaggle Secret HF_TOKEN có quyền Write vào fushinguyenex/GRIT.") from None

from huggingface_hub import HfApi
HUB_UPLOAD_RESULT = publish_trained_model(HfApi, kaggle_hf_token)


## 13. Lưu kết quả / archive / cleanup

Kiểm tra logs `smoke.log`, `smoke-resume.log`, `train.log`, `*-qwen3guard_vllm.log`, resource samples,
run manifest, checkpoint receipts và reload generation. Timing rollout/backward nằm trong metric logs
của verl; nếu thiếu guard timing hoặc allocator peak thì ghi là chưa đo, không tự suy diễn speedup.
Không có process dài hạn do notebook giữ sau `run_stage`; không gọi `pkill` theo tên hoặc `ray stop`
vì có thể tác động worker không thuộc run.

**Save Version → Save & Run All / lưu output phù hợp với phiên đang chạy**, xác nhận version/output
đã lưu xong trước khi dừng session. `/kaggle/working` không phải persistent volume. Để tiếp tục train,
cần lưu nguyên run manifest, model lock và checkpoint có optimizer/extra/data, không chỉ model export.
Archive mặc định tắt vì có thể nhân đôi disk usage. Model inference được upload Hugging Face ở cell trước; checkpoint resume vẫn cần lưu Kaggle output. Không upload Google Drive.

In [ ]:
summary = dict(run_id=RUN_ID, repo_commit=COMMIT,
    smoke_resume_passed=smoke_receipt.exists(),
    full_training_requested=RUN_FULL_TRAINING,
    checkpoints=[str(p.relative_to(OUT)) for p in complete_checkpoints(OUT / "smoke/checkpoints")
                 + complete_checkpoints(TRAIN_DIR)],
    reload_receipts=[str(p.relative_to(OUT)) for p in (OUT / "exports").glob("*/reload_passed.json")],
    guard_timing="See runner metrics; not inferred if absent",
    memory_measurement="Sampled nvidia-smi; inspect worker metrics for exact peak",
    hub_upload=load_json(OUT / "hub_upload.json") if (OUT / "hub_upload.json").exists() else None,
    save_version_required=True)
atomic_json(OUT / "run_summary.json", summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))
if CREATE_ARCHIVE:
    import tarfile
    archive = OUT.parent / (OUT.name + ".tar.gz")
    assert not archive.exists(), "Archive đã tồn tại; không tự ghi đè."
    estimated = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file())
    assert shutil.disk_usage(OUT).free > estimated + 2 * 2**30, "Không đủ disk để archive."
    with tarfile.open(archive, "w:gz") as tar:
        tar.add(OUT, arcname=OUT.name, filter=lambda info: None if info.name.endswith("runner.lock") else info)
    print("Archive:", archive)
print("Save Version và xác nhận output đã lưu:", OUT)


## Phạm vi kiểm chứng và nguồn đối chiếu

- Bản tạo notebook: JSON/schema, compile các cell và Python snippets; CPU fixtures cho checksum,
  bundle discovery, projector shape/provenance/math, metrics, checkpoint identity và command construction.
  Fixture chỉ kiểm chứng cơ chế kiểm tra, **không phải bundle thật hay training GPU**.
- Commit cũ bị audit chặn. Backend mới thêm ProjectedAdamW, FP16 scaler, skip zero/nonfinite,
  structured Guard reward và CLI overrides. Không có tuyên bố “chạy end-to-end”.
- Sau khi backend sửa: cần import stack, hai T4 không OOM, guard ready, projected metrics,
  ratio check, 2 steps + restart/resume thêm 1 step, checkpoint reload và generation trên Kaggle.
- Nếu OOM: kiểm tra memory log, giảm rollout concurrency/KV budget qua config backend đã hỗ trợ;
  không tự giảm rank projector, bỏ module hoặc chuyển LoRA. Không có worker/RPC tùy biến trong notebook.

Nguồn: [GRIT repo](https://github.com/namdeptraivcd/GRIT),
[Qwen3Guard: response moderation bằng chat template, structured labels](https://github.com/QwenLM/Qwen3Guard),
[vLLM 0.9.2 CUDA pins](https://github.com/vllm-project/vllm/blob/v0.9.2/requirements/cuda.txt).
Trong repo xem `WORKFLOW.md`, `scripts/run_grit_vllm.sh`, `scripts/prepare_verl_nspo_data.py`,
`scripts/nspo_vllm_reward.py` và `verl/verl/workers/actor/dp_actor.py`.